In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Mewakili komputer kuantum untuk Transpiler


<details>
<summary><b>Versi pakej</b></summary>

Kod pada halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan menggunakan versi ini atau yang lebih baharu.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Untuk menukar Circuit abstrak kepada Circuit ISA yang boleh berjalan pada QPU tertentu (unit pemprosesan kuantum), Transpiler memerlukan maklumat tertentu tentang QPU. Maklumat ini ditemui di dua tempat: objek `BackendV2` (atau `BackendV1` warisan) yang anda rancang untuk menghantar kerja, dan atribut `Target` backend.

- [`Target`](../api/qiskit/qiskit.transpiler.Target) mengandungi semua kekangan yang relevan peranti, seperti Gate asas natif, konektiviti Qubit, dan maklumat denyutan atau pemasaan.
- [`Backend`](../api/qiskit/qiskit.providers.BackendV2) memiliki `Target` secara lalai, mengandungi maklumat tambahan -- seperti [`InstructionScheduleMap`](https://docs.quantum.ibm.com/api/qiskit/1.4/qiskit.pulse.InstructionScheduleMap), dan menyediakan antara muka untuk menghantar kerja Circuit kuantum.

Anda juga boleh memberikan maklumat secara eksplisit untuk Transpiler gunakan, contohnya, jika anda mempunyai kes penggunaan tertentu, atau jika anda percaya maklumat ini akan membantu Transpiler menghasilkan Circuit yang lebih dioptimumkan.

Ketepatan di mana Transpiler menghasilkan Circuit yang paling sesuai untuk perkakasan tertentu bergantung pada berapa banyak maklumat yang dimiliki `Target` atau `Backend` tentang kekanannya.

> **Note:** Kerana banyak algoritma transpilasi asas adalah stokastik, tiada jaminan bahawa Circuit yang lebih baik akan ditemui.

Halaman ini menunjukkan beberapa contoh menghantar maklumat QPU kepada Transpiler. Contoh-contoh ini menggunakan target dari backend mock [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke).

<span id="default-config"></span>
## Konfigurasi lalai
Penggunaan paling mudah Transpiler adalah dengan menyediakan semua maklumat QPU dengan menyediakan `Backend` atau `Target`. Untuk memahami cara Transpiler berfungsi dengan lebih baik, bina Circuit dan transpilkannya dengan maklumat berbeza, seperti berikut.

Import perpustakaan yang diperlukan dan instantiasikan QPU:
Untuk menukar Circuit abstrak kepada Circuit ISA yang boleh berjalan pada pemproses tertentu, Transpiler memerlukan maklumat tertentu tentang pemproses. Biasanya, maklumat ini disimpan dalam [`Backend`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.Backend#backend) atau [`Target`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.Target#target) yang disediakan kepada Transpiler, dan tiada maklumat lanjut diperlukan. Walau bagaimanapun, anda juga boleh memberikan maklumat secara eksplisit untuk Transpiler gunakan, contohnya, jika anda mempunyai kes penggunaan tertentu, atau jika anda percaya maklumat ini akan membantu Transpiler menghasilkan Circuit yang lebih dioptimumkan.

Topik ini menunjukkan beberapa contoh menghantar maklumat kepada Transpiler. Contoh-contoh ini menggunakan target dari backend mock [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke).

In [1]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()
target = backend.target

Circuit contoh menggunakan instans [`efficient_su2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.efficient_su2) dari perpustakaan Circuit Qiskit.

In [2]:
from qiskit.circuit.library import efficient_su2

qc = efficient_su2(12, entanglement="circular", reps=1)

qc.draw("mpl")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/97f9acc1-ac53-4025-b413-485777932a9b-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/97f9acc1-ac53-4025-b413-485777932a9b-0.svg)

Contoh ini menggunakan tetapan lalai untuk transpilasi ke `target` `backend`, yang menyediakan semua maklumat yang diperlukan untuk menukar Circuit kepada yang akan berjalan pada backend.

In [3]:
from qiskit.transpiler import generate_preset_pass_manager

pass_manager = generate_preset_pass_manager(
    optimization_level=1, target=target, seed_transpiler=12345
)
qc_t_target = pass_manager.run(qc)
qc_t_target.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/4b81fb9d-d199-45c5-b119-c1f0b973afe9-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/4b81fb9d-d199-45c5-b119-c1f0b973afe9-0.svg)

Contoh ini digunakan dalam bahagian kemudian topik ini untuk menggambarkan bahawa peta gandeng dan Gate asas adalah maklumat penting yang perlu diberikan kepada Transpiler untuk pembinaan Circuit yang optimum. QPU biasanya boleh memilih tetapan lalai untuk maklumat lain yang tidak diberikan, seperti pemasaan dan penjadualan.

## Peta gandeng

Peta gandeng adalah graf yang menunjukkan Qubit mana yang disambungkan dan oleh itu mempunyai Gate dua-Qubit antara mereka. Kadang-kadang graf ini adalah berarah, bermakna Gate dua-Qubit hanya boleh pergi dalam satu arah. Walau bagaimanapun, Transpiler boleh sentiasa membalikkan arah Gate dengan menambah Gate satu-Qubit tambahan. Circuit kuantum abstrak boleh sentiasa diwakili pada graf ini, walaupun konektivitinya terhad, dengan memperkenalkan Gate SWAP untuk memindahkan maklumat kuantum.

Qubit dari Circuit abstrak kami dipanggil _Qubit maya_ dan yang pada peta gandeng adalah _Qubit fizikal_. Transpiler menyediakan pemetaan antara Qubit maya dan fizikal. Salah satu langkah pertama dalam transpilasi, peringkat _susun atur_, melakukan pemetaan ini.

> **Note:** Walaupun peringkat penghala saling berkait dengan peringkat _susun atur_ — yang memilih Qubit sebenar — secara lalai, topik ini menganggap mereka sebagai peringkat berasingan untuk kesederhanaan. Gabungan penghala dan susun atur dipanggil _pemetaan Qubit_. Ketahui lebih lanjut tentang peringkat-peringkat ini dalam topik [Peringkat Transpiler](transpiler-stages).

Hantar argumen kata kunci `coupling_map` untuk melihat kesannya pada Transpiler:

In [4]:
coupling_map = target.build_coupling_map()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, coupling_map=coupling_map, seed_transpiler=12345
)
qc_t_cm_lv0 = pass_manager.run(qc)
qc_t_cm_lv0.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/ec354bee-e06b-42ea-a117-6c1a9308ca73-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/ec354bee-e06b-42ea-a117-6c1a9308ca73-0.svg)

Seperti yang ditunjukkan di atas, beberapa Gate SWAP telah dimasukkan (setiap satu terdiri daripada tiga Gate CX), yang akan menyebabkan banyak ralat pada peranti semasa. Untuk melihat Qubit mana yang dipilih pada topologi Qubit sebenar, gunakan `plot_circuit_layout` dari Visualisasi Qiskit:

In [5]:
from qiskit.visualization import plot_circuit_layout

plot_circuit_layout(qc_t_cm_lv0, backend, view="physical")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/9be74535-ed36-4d51-afeb-ee53c3f8a046-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/9be74535-ed36-4d51-afeb-ee53c3f8a046-0.svg)

Ini menunjukkan bahawa Qubit maya 0-11 kami dipetakan secara trivial ke barisan Qubit fizikal 0-11. Mari kita kembali ke lalai (`optimization_level=1`), yang menggunakan `VF2Layout` jika sebarang penghala diperlukan.

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, coupling_map=coupling_map, seed_transpiler=12345
)
qc_t_cm_lv1 = pass_manager.run(qc)
qc_t_cm_lv1.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/8035fd05-f7cd-4151-b19a-4968202246e6-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/8035fd05-f7cd-4151-b19a-4968202246e6-0.svg)

Kini tiada Gate SWAP dimasukkan dan Qubit fizikal yang dipilih adalah sama apabila menggunakan kelas `target`.

In [7]:
from qiskit.visualization import plot_circuit_layout

plot_circuit_layout(qc_t_cm_lv1, backend, view="physical")

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/25d9fac3-abda-4b2d-81b4-351dc0772722-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/25d9fac3-abda-4b2d-81b4-351dc0772722-0.svg)

Kini susun atur adalah dalam bentuk cincin. Kerana susun atur ini menghormati konektiviti Circuit, tiada Gate SWAP, memberikan Circuit yang jauh lebih baik untuk pelaksanaan.

## Gate asas

Setiap komputer kuantum menyokong set arahan yang terhad, dipanggil _Gate asas_ nya. Setiap Gate dalam Circuit mesti diterjemahkan kepada elemen set ini. Set ini hendaklah terdiri daripada Gate satu- dan dua-Qubit yang menyediakan set Gate universal, bermakna sebarang operasi kuantum boleh didekomposisikan kepada Gate-gate tersebut. Ini dilakukan oleh [BasisTranslator](../api/qiskit/qiskit.transpiler.passes.BasisTranslator), dan Gate asas boleh ditentukan sebagai argumen kata kunci kepada Transpiler untuk menyediakan maklumat ini.

In [8]:
basis_gates = list(target.operation_names)
print(basis_gates)

['sx', 'switch_case', 'x', 'if_else', 'measure', 'for_loop', 'delay', 'ecr', 'id', 'reset', 'rz']


The default single-qubit gates on _ibm_sherbrooke_ are `rz`, `x`, and `sx`, and the default two-qubit gate is `ecr` (echoed cross-resonance). CX gates are constructed from `ecr` gates, so on some QPUs `ecr` is specified as the two-qubit basis gate, while on others `cx` is the default. The `ecr` gate is the _entangling_ part of the `cx` gate. In addition to the control gates, there are also `delay` and `measurement` instructions.


<Admonition type="note">
    QPUs have default basis gates, but you can choose whatever gates you want, as long as you provide the instruction or add pulse gates (see [Create transpiler passes.](custom-transpiler-pass)) The default basis gates are those that calibrations have been done for on the QPU, so no further instruction/pulse gates need to be provided. For example, on some QPUs `cx` is the default two-qubit gate and `ecr` on others. See the list of possible [native gates and operations](/docs/guides/qpu-information#native-gates) for more details.
</Admonition>

In [9]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    coupling_map=coupling_map,
    basis_gates=basis_gates,
    seed_transpiler=12345,
)
qc_t_cm_bg = pass_manager.run(qc)
qc_t_cm_bg.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/313e4743-0.svg" alt="Output of the previous code cell" />

Gate satu-Qubit lalai pada _ibm_sherbrooke_ adalah `rz`, `x`, dan `sx`, dan Gate dua-Qubit lalai adalah `ecr` (resonans silang gema). Gate CX dibina dari Gate `ecr`, jadi pada beberapa QPU `ecr` ditentukan sebagai Gate asas dua-Qubit, sementara pada yang lain `cx` adalah lalai. Gate `ecr` adalah bahagian _belitan_ Gate `cx`. Selain Gate kawalan, terdapat juga arahan `delay` dan `measurement`.

> **Note:** QPU mempunyai Gate asas lalai, tetapi anda boleh memilih Gate apa sahaja yang anda mahukan, selagi anda menyediakan arahan atau menambah Gate denyutan (lihat [Cipta lintasan Transpiler.](custom-transpiler-pass)) Gate asas lalai adalah yang telah dilakukan penentukuran pada QPU, jadi tiada arahan/Gate denyutan lanjut perlu disediakan. Contohnya, pada beberapa QPU `cx` adalah Gate dua-Qubit lalai dan `ecr` pada yang lain. Lihat senarai kemungkinan [Gate dan operasi natif](/guides/qpu-information#native-gates) untuk butiran lanjut.

In [10]:
target["ecr"][(1, 0)]

InstructionProperties(duration=5.333333333333332e-07, error=0.007494257741828603)

![Output of the previous code cell](../docs/images/guides/represent-quantum-computers/extracted-outputs/313e4743-0.svg)

Perhatikan bahawa objek `CXGate` telah didekomposisikan kepada Gate `ecr` dan Gate asas satu-Qubit.
## Kadar ralat peranti
Kelas `Target` boleh mengandungi maklumat tentang kadar ralat untuk operasi pada peranti.
Contohnya, kod berikut mendapatkan semula sifat-sifat Gate resonans silang gema (ECR) antara Qubit 1 dan 0 (perhatikan bahawa Gate ECR adalah berarah):

In [11]:
from qiskit.transpiler import Target
from qiskit.circuit.controlflow import IfElseOp, SwitchCaseOp, ForLoopOp

err_targ = Target.from_configuration(
    basis_gates=basis_gates,
    coupling_map=coupling_map,
    num_qubits=target.num_qubits,
    custom_name_mapping={
        "if_else": IfElseOp,
        "switch_case": SwitchCaseOp,
        "for_loop": ForLoopOp,
    },
)

for i, (op, qargs) in enumerate(target.instructions):
    if op.name in basis_gates:
        err_targ[op.name][qargs] = target.instruction_properties(i)

Transpile with our new target `err_targ` as the target:

In [12]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, target=err_targ, seed_transpiler=12345
)
qc_t_cm_bg_et = pass_manager.run(qc)
qc_t_cm_bg_et.draw("mpl", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/represent-quantum-computers/extracted-outputs/f1e270c4-e2cc-487e-a050-4180bc321b0b-0.svg" alt="Output of the previous code cell" />

Output memaparkan tempoh Gate (dalam saat) dan kadar ralatnya. Untuk mendedahkan maklumat ralat kepada Transpiler, bina model target dengan `basis_gates` dan `coupling_map` dari atas dan isi dengan nilai ralat dari backend `FakeSherbrooke`.